<a href="https://colab.research.google.com/github/husainal1/fpl-predictor-app/blob/main/Predicting_EPL_Player_Performance_2026_27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EPL / FPL player performance predictor (2026/27)

This predicts how many FPL points each player is likely to score in their next match, then builds the best squad you can afford.

A few notes on how it works this year:

It pulls the current 2026/27 data (players, prices, positions, fixtures and team strengths) live from the FPL API, so nothing about the season is written into the code.

It trains on the last few seasons of real match data, which means it still gives sensible numbers before the season has even started.

It works out on its own whether the season is underway. Before kickoff it leans on last season's form; once real games are played it starts folding them in every time you run it.

It also copes with players moving around. If someone switched clubs over the summer he's rated against his new team's strength and fixtures, and promoted clubs or brand-new signings who have never played in the league get a reasonable starting estimate instead of breaking things.

And it only uses what you would actually know before a match (recent form plus the upcoming opponent and venue), so the predictions reflect a real team-picking decision.

Just run the cells top to bottom. Everything works in Colab with no local setup.

## Setup

In [1]:
# One-time installs
!pip -q install pandas numpy requests tqdm scikit-learn xgboost pulp unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.8 MB/s eta 0:00:00


In [2]:
import warnings, time, unicodedata
import numpy as np, pandas as pd, requests
from tqdm.auto import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ------------------------------- CONFIG ------------------------------------
BASE     = "https://fantasy.premierleague.com/api"
ARCHIVE  = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data"
# Past seasons of match history to train on (most recent first is fine):
HIST_SEASONS   = ["2023-24", "2024-25", "2025-26"]
CURRENT_SEASON = "2026-27"

LAGS    = (1, 2, 3)      # look-back individual games
WINDOWS = (3, 5)         # rolling-form windows
SQUAD_BUDGET = 100.0     # FPL budget in millions

POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}   # FPL element_type -> position
VALID_POS = ["GK", "DEF", "MID", "FWD"]

def norm_name(s):
    """Accent- and case-insensitive key so we can match a player across seasons."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return " ".join(s.lower().split())

In [3]:
def get_json(url, retries=5, sleep=0.6):
    """GET JSON from the FPL API with a simple retry/backoff."""
    last = None
    for i in range(retries):
        try:
            r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            if r.status_code == 200:
                return r.json()
            last = f"HTTP {r.status_code}"
        except Exception as e:
            last = str(e)
        time.sleep(sleep * (i + 1))
    raise RuntimeError(f"Failed to fetch {url}: {last}")

## Live 2026/27 data

In [4]:
# ---- Live current-season data (this IS 2026/27 — pulled fresh every run) ----
boot     = get_json(f"{BASE}/bootstrap-static/")
elements = pd.DataFrame(boot["elements"])
teams_raw= pd.DataFrame(boot["teams"])
events   = pd.DataFrame(boot["events"])
fixtures = pd.DataFrame(get_json(f"{BASE}/fixtures/"))

# Season state: how many gameweeks have actually been played?
finished_gws = int(events["finished"].sum())
cur_row  = events[events["is_current"]]
next_row = events[events["is_next"]]
CURRENT_GW = int(cur_row["id"].iloc[0]) if len(cur_row) else 0
NEXT_GW    = int(next_row["id"].iloc[0]) if len(next_row) else (CURRENT_GW + 1 if CURRENT_GW else 1)
played_any = bool((pd.to_numeric(elements["minutes"], errors="coerce").fillna(0) > 0).any())
SEASON_STARTED = finished_gws > 0 or played_any
print(f"Season {CURRENT_SEASON}: {finished_gws} GW(s) finished | current GW={CURRENT_GW} | next GW={NEXT_GW}")
print("Pre-season cold start." if not SEASON_STARTED else "In-season: live data will be blended in.")

# id -> team name for the CURRENT season
team_name_now = dict(zip(teams_raw["id"], teams_raw["name"]))

# Current-season team strength from FPL's published ratings (available pre-season).
ts = teams_raw.copy()
ts["attack"]  = ts[["strength_attack_home",  "strength_attack_away"]].mean(axis=1)
ts["defence"] = ts[["strength_defence_home", "strength_defence_away"]].mean(axis=1)
def _z(x):
    x = pd.to_numeric(x, errors="coerce"); sd = x.std(ddof=0)
    return (x - x.mean()) / sd if sd and not np.isnan(sd) else x*0.0
ts["attack_z"]  = _z(ts["attack"])
ts["defence_z"] = _z(ts["defence"])   # higher = stronger defence
cur_strength = {r["name"]: (float(r["attack_z"]), float(r["defence_z"])) for _, r in ts.iterrows()}

# Current roster: who is in the league now, their price, position, club (source of truth).
players_now = pd.DataFrame({
    "player_id": elements["id"],
    "web_name":  elements["web_name"],
    "full_name": (elements["first_name"] + " " + elements["second_name"]),
    "team_name": elements["team"].map(team_name_now),
    "position":  elements["element_type"].map(POS_MAP),
    "price":     elements["now_cost"] / 10.0,
    "status":    elements["status"],
    "chance":    pd.to_numeric(elements.get("chance_of_playing_next_round"), errors="coerce"),
})
players_now = players_now[players_now["position"].isin(VALID_POS)].copy()
players_now = players_now[players_now["status"] != "u"].copy()
players_now["name_key"] = players_now["full_name"].map(norm_name)
print(f"{len(players_now)} outfield/keeper players in the current game.")
players_now.head()

Season 2026-27: 3 GW(s) finished | current GW=3 | next GW=4
In-season: live data will be blended in.
555 outfield/keeper players in the current game.


,player_id,web_name,full_name,team_name,position,price,status,chance,name_key
0,1,Raya,David Raya Martín,Arsenal,GK,6.0,a,NaN,david raya martin
1,2,Arrizabalaga,Kepa Arrizabalaga Revuelta,Arsenal,GK,4.9,a,NaN,kepa arrizabalaga revuelta
2,3,Meslier,Illan Meslier,Arsenal,GK,4.9,a,NaN,illan meslier
3,4,Gabriel,Gabriel dos Santos Magalhães,Arsenal,DEF,8.0,a,NaN,gabriel dos santos magalhaes
4,5,J.Timber,Jurriën Timber,Arsenal,DEF,6.5,d,25.0,jurrien timber


## Historical match data

In [5]:
# ---- Multi-season historical match data (community archive) ----
# Same columns the model needs, plus name/team/position so we can match across seasons.
KEEP = ["total_points","minutes","goals_scored","assists","ict_index","creativity",
        "influence","threat","expected_goals","expected_assists",
        "expected_goal_involvements","expected_goals_conceded",
        "team_h_score","team_a_score","was_home","kickoff_time","round"]

# opponent id -> name per season
mtl = pd.read_csv(f"{ARCHIVE}/master_team_list.csv")
opp_name_by_season = {(str(s), int(t)): n
                      for s, t, n in zip(mtl["season"], mtl["team"], mtl["team_name"])}

def load_season(season):
    df = pd.read_csv(f"{ARCHIVE}/{season}/gws/merged_gw.csv", encoding="utf-8-sig")
    for c in KEEP:                       # older seasons may miss a column
        if c not in df.columns: df[c] = np.nan
    out = df[["name","position","team","opponent_team","value"] + KEEP].copy()
    out["season"]   = season
    out["name_key"] = out["name"].map(norm_name)
    out["opp_name"] = [opp_name_by_season.get((season, int(o)), None)
                       if pd.notna(o) else None for o in out["opponent_team"]]
    return out

hist_archive = pd.concat([load_season(s) for s in tqdm(HIST_SEASONS, desc="seasons")],
                         ignore_index=True)
# normalise position labels (e.g. GKP -> GK) and drop non-players (managers)
hist_archive["position"] = hist_archive["position"].replace({"GKP": "GK", "AM": "MID"})
hist_archive = hist_archive[hist_archive["position"].isin(VALID_POS)].copy()
print("historical rows:", len(hist_archive), "| players:", hist_archive["name_key"].nunique())

seasons:   0%|          | 0/3 [00:00<?, ?it/s]

historical rows: 87087 | players: 1492


In [6]:
# ---- Multi-season historical match data (community archive) ----
# Same columns the model needs, plus name/team/position so we can match across seasons.
KEEP = ["total_points","minutes","goals_scored","assists","ict_index","creativity",
        "influence","threat","expected_goals","expected_assists",
        "expected_goal_involvements","expected_goals_conceded",
        "team_h_score","team_a_score","was_home","kickoff_time","round"]

# opponent id -> name per season
mtl = pd.read_csv(f"{ARCHIVE}/master_team_list.csv")
opp_name_by_season = {(str(s), int(t)): n
                      for s, t, n in zip(mtl["season"], mtl["team"], mtl["team_name"])}

def load_season(season):
    df = pd.read_csv(f"{ARCHIVE}/{season}/gws/merged_gw.csv", encoding="utf-8-sig")
    for c in KEEP:  # older seasons may miss a column
        if c not in df.columns: df[c] = np.nan
    out = df[["name","position","team","opponent_team","value"] + KEEP].copy()
    out["season"] = season
    out["name_key"] = out["name"].map(norm_name)
    out["opp_name"] = [opp_name_by_season.get((season, int(o)), None)
                       if pd.notna(o) else None for o in out["opponent_team"]]
    return out

hist_archive = pd.concat([load_season(s) for s in tqdm(HIST_SEASONS, desc="seasons")],
                         ignore_index=True)
# normalise position labels (e.g. GKP -> GK) and drop non-players (managers)
hist_archive["position"] = hist_archive["position"].replace({"GKP": "GK", "AM": "MID"})
hist_archive = hist_archive[hist_archive["position"].isin(VALID_POS)].copy()
print("historical rows:", len(hist_archive), "| players:", hist_archive["name_key"].nunique())

# ---- Current-season live match history (only exists once games are played) ----
def fetch_player_history(pid):
    j = get_json(f"{BASE}/element-summary/{pid}/")
    return pd.DataFrame(j.get("history", []))

live_rows = []
if SEASON_STARTED:
    pos_now  = dict(zip(players_now["player_id"], players_now["position"]))
    name_now = dict(zip(players_now["player_id"], players_now["full_name"]))
    price_now= dict(zip(players_now["player_id"], players_now["price"]))
    team_now = dict(zip(players_now["player_id"], players_now["team_name"]))   # <-- NEW
    for pid in tqdm(players_now["player_id"], desc="live history"):
        try:
            h = fetch_player_history(pid)
            if h.empty: continue
            h["name"] = name_now.get(pid)
            h["position"] = pos_now.get(pid)
            h["team"] = team_now.get(pid)      # <-- CHANGED (was None): real team so strength isn't zeroed
            h["value"] = price_now.get(pid, np.nan) * 10
            h["season"] = CURRENT_SEASON
            h["opp_name"] = h["opponent_team"].map(team_name_now)
            live_rows.append(h)
        except Exception:
            pass

if live_rows:
    live = pd.concat(live_rows, ignore_index=True)
    for c in KEEP:
        if c not in live.columns: live[c] = np.nan
    live["name_key"] = live["name"].map(norm_name)
    live = live[["name","position","team","opponent_team","value","season","name_key","opp_name"]+KEEP]
    hist = pd.concat([hist_archive, live], ignore_index=True)
else:
    hist = hist_archive.copy()
print("combined match rows:", len(hist))

seasons:   0%|          | 0/3 [00:00<?, ?it/s]

historical rows: 87087 | players: 1492


live history:   0%|          | 0/555 [00:00<?, ?it/s]

combined match rows: 88681


## Team strength and context

In [7]:
# ---- Team strength per (season, team) derived from actual match scorelines ----
h = hist.copy()
h["gf"] = np.where(h["was_home"] == True, h["team_h_score"], h["team_a_score"])
h["ga"] = np.where(h["was_home"] == True, h["team_a_score"], h["team_h_score"])
tm = (h.dropna(subset=["team","gf","ga"])
        .drop_duplicates(["season","team","round"])
        .groupby(["season","team"])
        .agg(gf=("gf","mean"), ga=("ga","mean")).reset_index())
tm["attack_z"]  = tm.groupby("season")["gf"].transform(lambda x: (x-x.mean())/(x.std(ddof=0) or 1))
tm["defence_z"] = tm.groupby("season")["ga"].transform(lambda x: -(x-x.mean())/(x.std(ddof=0) or 1))
hist_strength = {(r["season"], r["team"]): (float(r["attack_z"]), float(r["defence_z"]))
                 for _, r in tm.iterrows()}

def team_strength(season, team_name):
    """Attack/defence z-score for a club in a season. Current season & unknowns
    fall back to the FPL published ratings, then to league-average (0,0)."""
    if season == CURRENT_SEASON and team_name in cur_strength:
        return cur_strength[team_name]
    if (season, team_name) in hist_strength:
        return hist_strength[(season, team_name)]
    if team_name in cur_strength:            # e.g. promoted club seen only in current data
        return cur_strength[team_name]
    return (0.0, 0.0)

print("teams with a derived strength rating:", len(hist_strength))

teams with a derived strength rating: 80


## Features

In [8]:
# ---- Feature engineering: predict a match from PRE-match info only ----
BASE_COLS = ["total_points","minutes","goals_scored","assists","ict_index","creativity",
             "influence","threat","expected_goals","expected_assists","expected_goal_involvements"]

def add_features(df):
    """df: match rows for MANY players. Returns rows with leakage-free form features,
    this-match opponent/venue context, position, price. Target = this match's points."""
    df = df.copy()
    df["kickoff_time"] = pd.to_datetime(df["kickoff_time"], errors="coerce", utc=True)
    # order each player's games chronologically across seasons
    df = df.sort_values(["name_key","kickoff_time","season","round"]).reset_index(drop=True)
    g = df.groupby("name_key", group_keys=False)

    for col in BASE_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        for L in LAGS:
            df[f"{col}_lag{L}"] = g[col].shift(L)
        for W in WINDOWS:
            df[f"{col}_roll{W}_mean"] = g[col].shift(1).rolling(W).mean()
    df["played_last"]      = g["minutes"].shift(1).fillna(0).gt(0).astype(int)
    df["played_last3_pct"] = g["minutes"].shift(1).rolling(3).apply(lambda x: np.mean(x>0), raw=True)

    # this-match team/opponent context (known before kickoff)
    st  = df.apply(lambda r: team_strength(r["season"], r["team"]),     axis=1)
    ost = df.apply(lambda r: team_strength(r["season"], r["opp_name"]), axis=1)
    df["team_attack_z"], df["team_defence_z"] = zip(*st)
    df["opp_attack_z"],  df["opp_defence_z"]  = zip(*ost)
    df["attack_vs_oppdef"] = df["team_attack_z"] - df["opp_defence_z"]
    df["def_vs_oppatt"]    = df["team_defence_z"] - df["opp_attack_z"]
    df["is_home"] = (df["was_home"] == True).astype(int)

    for p in VALID_POS:
        df[f"is_{p}"] = (df["position"] == p).astype(int)
    df["price"] = pd.to_numeric(df["value"], errors="coerce") / 10.0
    df["month"] = df["kickoff_time"].dt.month.fillna(0).astype(int)
    return df

fe = add_features(hist)
fe["y"] = pd.to_numeric(fe["total_points"], errors="coerce")

FEATURES = ([f"{c}_lag{L}" for c in BASE_COLS for L in LAGS] +
            [f"{c}_roll{W}_mean" for c in BASE_COLS for W in WINDOWS] +
            ["played_last","played_last3_pct","team_attack_z","team_defence_z",
             "opp_attack_z","opp_defence_z","attack_vs_oppdef","def_vs_oppatt",
             "is_home","price","month"] + [f"is_{p}" for p in VALID_POS])
print("engineered rows:", len(fe), "| features:", len(FEATURES))

engineered rows: 88681 | features: 70


## Train and check the model

In [10]:
# ---- Train / validate (grouped by player so no player leaks across folds) ----
train = fe.dropna(subset=["y","minutes_lag1","total_points_lag1"]).copy()
X = train[FEATURES].fillna(0.0)
y = train["y"].astype(float)
groups = train["name_key"]
baseline = train["total_points_roll3_mean"].fillna(train["total_points_lag1"]).fillna(0)

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(train))
for tr, va in gkf.split(X, y, groups):
    m = XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                     random_state=42, n_jobs=-1, tree_method="hist")
    m.fit(X.iloc[tr], y.iloc[tr], verbose=False)
    oof[va] = m.predict(X.iloc[va])
print("Rows trained on : %d" % len(train))
print("Model MAE       : %.3f" % mean_absolute_error(y, oof))
print("Baseline MAE    : %.3f  (3-game points average)" % mean_absolute_error(y, baseline))

final_model = XGBRegressor(n_estimators=800, learning_rate=0.04, max_depth=6,
                           subsample=0.9, colsample_bytree=0.9, min_child_weight=5,
                           random_state=42, n_jobs=-1, tree_method="hist")
final_model.fit(X, y, verbose=False)

final_model.save_model('fpl_model.json')                      # download from Files panel
from google.colab import drive
drive.mount('/content/drive')
final_model.save_model('/content/drive/MyDrive/fpl_model.json')
print("Saved fpl_model.json")

Rows trained on : 87061
Model MAE       : 1.000
Baseline MAE    : 1.069  (3-game points average)
Mounted at /content/drive
Saved fpl_model.json


In [11]:
from google.colab import userdata
TOKEN  = userdata.get('GITHUB_TOKEN')
REPO   = "husainal1/fpl-predictor-app"
BRANCH = "main"

final_model.save_model('fpl_model.json')

!rm -rf _deploy && git clone -q https://{TOKEN}@github.com/{REPO}.git _deploy
!cp fpl_model.json _deploy/backend/
!cd _deploy && git rm -q -f backend/fpl_start_model.json backend/fpl_pts_start_model.json 2>/dev/null; \
  git config user.email "colab@local" && git config user.name "colab" \
  && git add -A && git commit -q -m "Revert to single model" \
  && git push -q origin {BRANCH} && echo "PUSHED -> Render will redeploy" || echo "nothing to push / check token"

PUSHED -> Render will redeploy


## Predict the next gameweek

In [12]:
# ---- Predict the upcoming gameweek for every current player ----
FORM_COLS = ([f"{c}_lag{L}" for c in BASE_COLS for L in LAGS] +
             [f"{c}_roll{W}_mean" for c in BASE_COLS for W in WINDOWS] +
             ["played_last", "played_last3_pct"])

def latest_form(df):
    df = df.copy()
    df["kickoff_time"] = pd.to_datetime(df["kickoff_time"], errors="coerce", utc=True)
    df = df.sort_values(["name_key", "kickoff_time", "season", "round"])
    key = df["name_key"]
    for col in BASE_COLS:
        s = pd.to_numeric(df[col], errors="coerce")
        for L in LAGS:
            df[f"{col}_lag{L}"] = s.groupby(key).shift(L - 1)
        for W in WINDOWS:
            df[f"{col}_roll{W}_mean"] = s.groupby(key).transform(lambda x: x.rolling(W).mean())
    mins = pd.to_numeric(df["minutes"], errors="coerce")
    df["played_last"]      = (mins > 0).astype(int)
    df["played_last3_pct"] = (mins > 0).groupby(key).transform(lambda x: x.rolling(3).mean())
    return df.groupby("name_key").tail(1)

recent = latest_form(hist)

fx = fixtures[(fixtures["event"] == NEXT_GW)].copy()
next_opp = {}
for _, r in fx.iterrows():
    h_, a_ = team_name_now.get(r["team_h"]), team_name_now.get(r["team_a"])
    if h_: next_opp[h_] = (a_, 1)
    if a_: next_opp[a_] = (h_, 0)

def build_pred_rows():
    r = recent.set_index("name_key")
    out = []
    for _, p in players_now.iterrows():
        row = {"player_id": p["player_id"], "web_name": p["web_name"],
               "team_name": p["team_name"], "position": p["position"], "price": p["price"]}
        opp, home = next_opp.get(p["team_name"], (None, 1))
        row["is_home"] = home
        ta, td = team_strength(CURRENT_SEASON, p["team_name"])
        oa, od = team_strength(CURRENT_SEASON, opp)
        row["team_attack_z"], row["team_defence_z"] = ta, td
        row["opp_attack_z"],  row["opp_defence_z"]  = oa, od
        row["attack_vs_oppdef"] = ta - od
        row["def_vs_oppatt"]    = td - oa
        row["month"] = pd.Timestamp.utcnow().month
        for pos in VALID_POS: row[f"is_{pos}"] = int(p["position"] == pos)
        if p["name_key"] in r.index:
            rr = r.loc[p["name_key"]]
            rr = rr.iloc[0] if isinstance(rr, pd.DataFrame) else rr
            for c in FORM_COLS:
                if c in rr.index: row[c] = rr[c]
            row["is_new"] = 0
        else:
            row["is_new"] = 1
        out.append(row)
    return pd.DataFrame(out)

pred = build_pred_rows()
for f in FEATURES:
    if f not in pred.columns: pred[f] = 0.0
pred[FEATURES] = pred[FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0.0)
pred["pred_points"] = final_model.predict(pred[FEATURES])
pred.loc[players_now.set_index("player_id").reindex(pred["player_id"])["status"].values != "a", "pred_points"] *= 0.4

# ---- set-piece / penalty bump ----
def _sp_col(name):
    return pd.to_numeric(elements[name], errors="coerce") if name in elements.columns else pd.Series([np.nan]*len(elements))
def _sp_bump(pen, dfk, ck):
    b = 0.0
    if pen == 1: b += 0.20
    elif pen == 2: b += 0.12
    if dfk == 1: b += 0.08
    if ck == 1: b += 0.06
    return b
_bump = {int(i): _sp_bump(pn, df, c) for i, pn, df, c in
         zip(elements["id"], _sp_col("penalties_order"),
             _sp_col("direct_freekicks_order"), _sp_col("corners_and_indirect_freekicks_order"))}
pred["pred_points"] = [pp + (_bump.get(int(pid), 0.0) if pp > 1.5 else 0.0)
                       for pp, pid in zip(pred["pred_points"], pred["player_id"])]
show = pred.sort_values("pred_points", ascending=False)
print(f"Top predicted scorers for GW{NEXT_GW}:")
show[["web_name","team_name","position","price","pred_points"]].head(20).round(2)

Top predicted scorers for GW4:


,web_name,team_name,position,price,pred_points
409,Haaland,Man City,FWD,15.5,7.99
148,Palmer,Chelsea,MID,9.6,6.39
433,B.Fernandes,Man Utd,MID,12.0,6.18
434,Mbeumo,Man Utd,MID,7.9,5.73
11,Saka,Arsenal,MID,9.5,5.60
396,Guéhi,Man City,DEF,6.0,5.51
155,João Pedro,Chelsea,FWD,7.7,5.50
404,Semenyo,Man City,MID,8.4,5.42
197,Mitchell,Crystal Palace,DEF,4.5,5.23
101,Thiago,Brentford,FWD,7.9,5.09



## Optimal squad

In [13]:
# ---- Optimal squad for the upcoming GW (clean, parameterised, no hardcoding) ----
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, value, LpBinary, PULP_CBC_CMD

def optimize_squad(pred_df, budget=SQUAD_BUDGET, max_per_club=3,
                   squad=(2,5,5,3), starters=11):
    """Pick 15 (2 GK, 5 DEF, 5 MID, 3 FWD) under budget & <=3 per club, then the best
    legal XI + captain (points doubled). Returns (squad_df, xi_ids, captain_id)."""
    d = pred_df.reset_index(drop=True).copy()
    P = range(len(d))
    pick  = LpVariable.dicts("pick",  P, cat=LpBinary)   # in 15-man squad
    start = LpVariable.dicts("start", P, cat=LpBinary)   # in starting XI
    cap   = LpVariable.dicts("cap",   P, cat=LpBinary)   # captain
    prob = LpProblem("fpl", LpMaximize)
    prob += lpSum((start[i] + cap[i]) * d.loc[i, "pred_points"] for i in P)

    prob += lpSum(pick[i] for i in P) == sum(squad)
    prob += lpSum(pick[i] * d.loc[i, "price"] for i in P) <= budget
    prob += lpSum(start[i] for i in P) == starters
    prob += lpSum(cap[i] for i in P) == 1
    need = dict(zip(VALID_POS, squad))
    xi_min = {"GK":1, "DEF":3, "MID":2, "FWD":1}   # valid FPL formation bounds
    xi_max = {"GK":1, "DEF":5, "MID":5, "FWD":3}
    for pos in VALID_POS:
        idx = [i for i in P if d.loc[i, "position"] == pos]
        prob += lpSum(pick[i]  for i in idx) == need[pos]
        prob += lpSum(start[i] for i in idx) >= xi_min[pos]
        prob += lpSum(start[i] for i in idx) <= xi_max[pos]
    for club in d["team_name"].unique():
        idx = [i for i in P if d.loc[i, "team_name"] == club]
        prob += lpSum(pick[i] for i in idx) <= max_per_club
    for i in P:
        prob += start[i] <= pick[i]
        prob += cap[i]   <= start[i]

    prob.solve(PULP_CBC_CMD(msg=0))
    d["in_squad"] = [int(value(pick[i]) or 0) for i in P]
    d["is_start"] = [int(value(start[i]) or 0) for i in P]
    d["is_cap"]   = [int(value(cap[i]) or 0) for i in P]
    sq = d[d["in_squad"] == 1].sort_values(["is_start","position"], ascending=[False, True])
    return sq, sq[sq["is_start"]==1]["player_id"].tolist(), sq[sq["is_cap"]==1]["player_id"].tolist()

squad, xi, captain = optimize_squad(pred)
cap_name = squad[squad["is_cap"]==1]["web_name"].iloc[0] if len(squad[squad["is_cap"]==1]) else "-"
print(f"Optimal 15 for GW{NEXT_GW}  |  cost "
      f"{squad['price'].sum():.1f}m  |  captain: {cap_name}")
cols = ["web_name","team_name","position","price","pred_points","is_start","is_cap"]
squad[cols].round(2).reset_index(drop=True)

Optimal 15 for GW4  |  cost 99.9m  |  captain: Haaland


,web_name,team_name,position,price,pred_points,is_start,is_cap
0,Mitchell,Crystal Palace,DEF,4.5,5.23,1,0
1,Castagne,Fulham,DEF,4.5,4.97,1,0
2,Guéhi,Man City,DEF,6.0,5.51,1,0
3,Thiago,Brentford,FWD,7.9,5.09,1,0
4,João Pedro,Chelsea,FWD,7.7,5.50,1,0
5,Haaland,Man City,FWD,15.5,7.99,1,1
6,Pickford,Everton,GK,5.5,3.77,1,0
7,Palmer,Chelsea,MID,9.6,6.39,1,0
8,Semenyo,Man City,MID,8.4,5.42,1,0
9,Mbeumo,Man Utd,MID,7.9,5.73,1,0


## Best value and differentials

Who gives you the most predicted points per million, plus the good picks that barely anyone owns yet. The first table sorts everyone by value (points per million). The second looks for low-owned players who are still projected to do well, which is usually where differentials hide.

In [14]:
# Best value (predicted points per million) and low-owned differentials.
# Uses pred (from the prediction step) plus live ownership from the API.
own = dict(zip(elements['id'], pd.to_numeric(elements['selected_by_percent'], errors='coerce')))
val = pred.copy()
val['owned_pct'] = val['player_id'].map(own)
val['pts_per_million'] = (val['pred_points'] / val['price']).round(2)
elig = val[val['pred_points'] > 0]

print('Best value (points per million, min 3.0 predicted pts):')
best_value = elig[elig['pred_points'] >= 3.0].sort_values('pts_per_million', ascending=False)
display(best_value[['web_name','team_name','position','price','pred_points','pts_per_million','owned_pct']].head(15).round(2))

print()
print('Differentials (projected 3.5+ pts, owned by under 10%):')
diffs = elig[(elig['owned_pct'] < 10) & (elig['pred_points'] >= 3.5)].sort_values('pred_points', ascending=False)
display(diffs[['web_name','team_name','position','price','pred_points','owned_pct']].head(15).round(2))

Best value (points per million, min 3.0 predicted pts):


,web_name,team_name,position,price,pred_points,pts_per_million,owned_pct
197,Mitchell,Crystal Palace,DEF,4.5,5.23,1.16,5.6
251,Castagne,Fulham,DEF,4.5,4.97,1.10,0.9
306,Diop,Ipswich Town,DEF,4.0,4.05,1.01,14.9
396,Guéhi,Man City,DEF,6.0,5.51,0.92,18.7
312,O'Shea,Ipswich Town,DEF,4.0,3.56,0.89,3.1
168,van Ewijk,Coventry City,DEF,4.0,3.51,0.88,11.2
276,Ajayi,Hull City,DEF,4.1,3.51,0.86,11.0
167,Thomas,Coventry City,DEF,4.0,3.40,0.85,8.7
174,Amenda,Coventry City,DEF,4.0,3.39,0.85,0.8
109,Boscagli,Brighton,DEF,4.5,3.82,0.85,0.3



Differentials (projected 3.5+ pts, owned by under 10%):


,web_name,team_name,position,price,pred_points,owned_pct
197,Mitchell,Crystal Palace,DEF,4.5,5.23,5.6
251,Castagne,Fulham,DEF,4.5,4.97,0.9
374,Wirtz,Liverpool,MID,7.4,4.76,8.5
435,Cunha,Man Utd,MID,7.9,4.59,7.1
540,E.Le Fée,Sunderland,MID,5.9,4.52,5.1
241,Barry,Everton,FWD,5.6,4.45,5.5
263,Gonzalo,Fulham,FWD,6.0,4.42,4.3
341,Stach,Leeds,MID,6.0,4.41,2.8
486,Gibbs-White,Nott'm Forest,MID,7.9,4.35,9.4
44,N.Jackson,Aston Villa,FWD,6.5,4.28,1.1


## Fixture-run planner (next few gameweeks)

FPL is a multi-week game, so this scores each player over the next few gameweeks instead of just one. It adds up the predicted points across their upcoming fixtures, handling double and blank gameweeks along the way. It is a better guide than a single week to who is actually worth bringing in.

In [15]:
# Score each player over the next few gameweeks, not just the next one.
HORIZON = 5
r_idx = recent.set_index('name_key')

def _row_for(p, opp, home, gw):
    row = {'player_id': p['player_id'], 'web_name': p['web_name'], 'team_name': p['team_name'],
           'position': p['position'], 'price': p['price'], 'gw': gw, 'is_home': home,
           'month': pd.Timestamp.utcnow().month}
    ta, td = team_strength(CURRENT_SEASON, p['team_name'])
    oa, od = team_strength(CURRENT_SEASON, opp)
    row['team_attack_z'], row['team_defence_z'] = ta, td
    row['opp_attack_z'], row['opp_defence_z'] = oa, od
    row['attack_vs_oppdef'] = ta - od
    row['def_vs_oppatt'] = td - oa
    for pos in VALID_POS:
        row['is_' + pos] = int(p['position'] == pos)
    if p['name_key'] in r_idx.index:
        rr = r_idx.loc[p['name_key']]
        rr = rr.iloc[0] if isinstance(rr, pd.DataFrame) else rr
        for c in FORM_COLS:
            if c in rr.index:
                row[c] = rr[c]
    return row

def predict_gw(gw):
    fx = fixtures[fixtures['event'] == gw]
    opp = {}
    for _, r in fx.iterrows():
        h = team_name_now.get(r['team_h']); a = team_name_now.get(r['team_a'])
        if h: opp.setdefault(h, []).append((a, 1))
        if a: opp.setdefault(a, []).append((h, 0))
    rows = [_row_for(p, o, hm, gw) for _, p in players_now.iterrows() for o, hm in opp.get(p['team_name'], [])]
    if not rows:
        return pd.DataFrame()
    pf = pd.DataFrame(rows)
    for f in FEATURES:
        if f not in pf.columns: pf[f] = 0.0
    pf[FEATURES] = pf[FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    pf['pred_points'] = final_model.predict(pf[FEATURES])
    pf['pred_points'] = [pp + (_bump.get(int(pid), 0.0) if pp > 1.5 else 0.0) for pp, pid in zip(pf['pred_points'], pf['player_id'])]
    return pf[['player_id','web_name','team_name','position','price','gw','pred_points']]

gws = list(range(NEXT_GW, min(NEXT_GW + HORIZON, 39)))
allgw = pd.concat([predict_gw(g) for g in gws], ignore_index=True)
run = (allgw.groupby(['player_id','web_name','team_name','position','price'])
       .agg(games=('gw','nunique'), horizon_points=('pred_points','sum')).reset_index())
run['per_game'] = (run['horizon_points'] / run['games']).round(2)
run = run.sort_values('horizon_points', ascending=False)
print('Best fixture runs over GW', gws[0], 'to', gws[-1], '(total predicted points):')
display(run[['web_name','team_name','position','price','games','horizon_points','per_game']].head(20).round(2))

Best fixture runs over GW 4 to 8 (total predicted points):


,web_name,team_name,position,price,games,horizon_points,per_game
340,Haaland,Man City,FWD,15.5,5,40.87,8.17
132,Palmer,Chelsea,MID,9.6,5,31.15,6.23
353,B.Fernandes,Man Utd,MID,12.0,5,28.78,5.76
11,Saka,Arsenal,MID,9.5,5,28.54,5.71
324,Guéhi,Man City,DEF,6.0,5,28.27,5.65
332,Semenyo,Man City,MID,8.4,5,27.84,5.57
354,Mbeumo,Man Utd,MID,7.9,5,27.02,5.40
93,Thiago,Brentford,FWD,7.9,5,26.71,5.34
142,João Pedro,Chelsea,FWD,7.7,5,26.60,5.32
218,Castagne,Fulham,DEF,4.5,5,26.18,5.24


## Where this is heading

Right now this is just the model. The plan from here is to turn it into something people can actually use: a simple web app where you look up a player and get a plain-english take on why he is a good pick this week, backed by a small public API and a bit of usage tracking so we can see what is landing.

The code above is written with that in mind. `pred` gives you the per-player points and `optimize_squad()` gives you the squad, and both can sit behind an API without much extra work.